In [23]:
# Import packages
from typing import Literal
from langchain.agents import AgentState, create_agent
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import NotRequired
import os
from langchain.tools import tool
import time
from dotenv import load_dotenv


In [24]:
load_dotenv()

# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT')
os.environ['DEEPSEEK_API_KEY'] = os.getenv('DEEPSEEK_API_KEY')

In [25]:

# 1. Define state with active_agent tracker
class MultiAgentState(AgentState):
    active_agent: NotRequired[str]



Fundamentally, a handover mechanism are sets of tools that return a command to update the state, triggering a transition to 
a new step or agent.

In [26]:

# 2. Create handoff tools
@tool
def transfer_to_it(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the IT agent."""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(        # Tool message is needed to complete the request-response cycle. Without it, the conversation history becomes malformed.
        content="Transferred to IT agent from support agent",
        tool_call_id=runtime.tool_call_id,
    )
    return Command(
        goto="it_agent",
        update={
            "active_agent": "it_agent",
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,
    )


In [27]:
@tool
def transfer_to_hr(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the HR agent."""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(        # Tool message is needed to complete the request-response cycle. Without it, the conversation history becomes malformed.
        content="Transferred to HR agent from support agent",
        tool_call_id=runtime.tool_call_id,
    )
    return Command(
        goto="hr_agent",
        update={
            "active_agent": "hr_agent",
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,
    )


In [28]:
@tool
def transfer_to_support(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the support agent."""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(
        content="Transferred to support agent from IT or HR agent",
        tool_call_id=runtime.tool_call_id,
    )
    return Command(
        goto="support_agent",
        update={
            "active_agent": "support_agent",
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,
    )


In [29]:
# 3. Create agents with handoff tools
it_agent = create_agent(
    model = "deepseek:deepseek-v4-flash",
    tools=[transfer_to_support],
    system_prompt="You are an IT support agent. Your first response should always be 'Have you tried turning it off and on again?' and then you MUST call the transfer_to_support tool to return the conversation to the support agent.",
)

hr_agent = create_agent(
    model = "deepseek:deepseek-v4-flash",
    tools=[transfer_to_support],
    system_prompt="You are an HR support agent. Your job is to be as unhelpful as possible. If there are any complaints, say that there is nothing that can be done, and then you MUST call the transfer_to_support tool to return the conversation to the support agent.",
)

support_agent = create_agent(
    model = "deepseek:deepseek-v4-flash",
    tools=[transfer_to_hr, transfer_to_it],
    system_prompt="You are a support agent and you help with support questions by reaching out to the appropriate agent. For help with IT issues, call the transfer_to_it tool. If asked about HR related topics, call the transfer_to_hr tool. If the advice returned from IT is unhelpful, call the transfer_to_hr tool to escalate to HR.",
)


In [30]:


# 4. Create agent nodes that invoke the agents
def call_it_agent(state: MultiAgentState) -> Command:
    """Node that calls the IT agent."""
    response = it_agent.invoke(state)
    return response

def call_hr_agent(state: MultiAgentState) -> Command:
    """Node that calls the HR agent."""
    response = hr_agent.invoke(state)
    return response

def call_support_agent(state: MultiAgentState) -> Command:
    """Node that calls the support agent."""
    response = support_agent.invoke(state)
    return response



In [31]:

# 5. Create router that checks if we should end or continue
def route_after_agent(state):
    messages = state.get("messages", [])

    # If the agent finished with a plain answer (no handoff tool call), we're done
    if messages and isinstance(messages[-1], AIMessage) and not messages[-1].tool_calls:
        return "__end__"

    # Otherwise route to the agent named by the handoff tool
    active = state.get("active_agent")
    return active if active else "support_agent"

def route_initial(
    state: MultiAgentState,
) -> Literal["it_agent", "hr_agent", "support_agent"]:
    """Route to the active agent based on state, default to support agent."""
    return state.get("active_agent") or "support_agent"



In [32]:
prompt_message = """Hi, I'm having trouble with my account login. Can you help?"""

In [33]:

# 6. Build the graph
builder = StateGraph(MultiAgentState)
builder.add_node("it_agent", call_it_agent)
builder.add_node("support_agent", call_support_agent)
builder.add_node("hr_agent", call_hr_agent)

# Start with conditional routing based on initial active_agent
builder.add_conditional_edges(START, route_initial, ["it_agent","hr_agent", "support_agent"])

# After each agent, check if we should end or route to another agent
builder.add_conditional_edges(
    "it_agent", route_after_agent, ["support_agent", END]
)

builder.add_conditional_edges(
    "hr_agent", route_after_agent, ["support_agent", END]
)

builder.add_conditional_edges(
    "support_agent", route_after_agent, ["it_agent", "hr_agent", END]
)

graph = builder.compile()
result = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"{prompt_message}",
            }
        ]
    }
)

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi, I'm having trouble with my account login. Can you help?
================================== Ai Message ==================================
Tool Calls:
  transfer_to_it (call_00_Ml4FY6yxrEIuVjUkiGSh2365)
 Call ID: call_00_Ml4FY6yxrEIuVjUkiGSh2365
  Args:
================================= Tool Message =================================
Name: transfer_to_it

Transferred to IT agent from support agent
================================== Ai Message ==================================

Have you tried turning it off and on again?
Tool Calls:
  transfer_to_support (call_00_1Z8vmqfNWvhE4zpqrrVN0083)
 Call ID: call_00_1Z8vmqfNWvhE4zpqrrVN0083
  Args:
================================= Tool Message =================================
Name: transfer_to_support

Transferred to support agent from IT or HR agent
================================== Ai Message ==================================

It looks like the initial IT gu